In [1]:
from google.colab import files

uploaded = files.upload()


Saving paimana_construction_bi_dataset_v2.csv to paimana_construction_bi_dataset_v2.csv


In [2]:
import pandas as pd

In [3]:
#Load the uploaded dataset

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

In [4]:
#Display the dataset dimensions

print("Dataset shape:", df.shape)

Dataset shape: (500, 14)


In [5]:
#Display column names

print("\nColumn names:")
print(df.columns.tolist())


Column names:
['Project_ID', 'Project_Name', 'Sector', 'Implementing_Agency', 'State', 'Original_Cost_Cr', 'Revised_Cost_Cr', 'Cost_Overrun_Cr', 'Original_Target_Date', 'Time_Extension_Days', 'Revised_Target_Date', 'Physical_Progress_Pct', 'Financial_Progress_Pct', 'Primary_Bottleneck']


In [6]:
#Display the first five rows

print("\nFirst five records:")
display(df.head())


First five records:


,Project_ID,Project_Name,Sector,Implementing_Agency,State,Original_Cost_Cr,Revised_Cost_Cr,Cost_Overrun_Cr,Original_Target_Date,Time_Extension_Days,Revised_Target_Date,Physical_Progress_Pct,Financial_Progress_Pct,Primary_Bottleneck
0,PRJ0001,Mega Infrastructure Development Phase 1,Real Estate,BMRCL,Karnataka,1664.68,1831.15,166.47,10-09-2023,236,03-05-2024,62,75,Contractual Dispute
1,PRJ0002,Mega Infrastructure Development Phase 2,Bridges & Tunnels,RVNL,Karnataka,1677.89,1677.89,0.00,03-11-2021,0,03-11-2021,71,79,NaN
2,PRJ0003,Mega Infrastructure Development Phase 3,Roads & Highways,NHAI,Tamil Nadu,1007.04,1258.80,251.76,12-01-2022,412,28-02-2023,50,60,Land Acquisition
3,PRJ0004,Mega Infrastructure Development Phase 4,Urban Public Transport,NHAI,Tamil Nadu,1467.59,1467.59,0.00,12-05-2023,0,12-05-2023,66,67,NaN
4,PRJ0005,Mega Infrastructure Development Phase 5,Construction,NHAI,Delhi,1280.50,2304.90,1024.40,01-02-2022,102,14-05-2022,69,60,Utility Shifting


In [7]:
#print("\nMissing values:")
display(df.isnull().sum())

,0
Project_ID,0
Project_Name,0
Sector,0
Implementing_Agency,0
State,0
Original_Cost_Cr,0
Revised_Cost_Cr,0
Cost_Overrun_Cr,0
Original_Target_Date,0
Time_Extension_Days,0


In [8]:
#Check data types

print("\nData types:")
display(df.dtypes)


Data types:


,0
Project_ID,object
Project_Name,object
Sector,object
Implementing_Agency,object
State,object
Original_Cost_Cr,float64
Revised_Cost_Cr,float64
Cost_Overrun_Cr,float64
Original_Target_Date,object
Time_Extension_Days,int64


In [9]:
#Check the cost-related columns

cost_columns = [
"Original_Cost_Cr",
"Revised_Cost_Cr",
"Cost_Overrun_Cr"
]

display(df[cost_columns].describe())

,Original_Cost_Cr,Revised_Cost_Cr,Cost_Overrun_Cr
count,500.000000,500.000000,500.00000
mean,1339.056660,1534.812100,195.75544
std,677.834311,838.461409,325.56442
min,150.560000,150.560000,0.00000
25%,746.505000,842.212500,0.00000
50%,1378.585000,1533.005000,74.68500
75%,1908.257500,2139.220000,229.56500
max,2491.390000,4471.810000,1987.47000


In [10]:
#Verify the cost-overrun calculation

calculated_overrun = (
df["Revised_Cost_Cr"] - df["Original_Cost_Cr"]
)

difference = (
df["Cost_Overrun_Cr"] - calculated_overrun
).abs()

print("Rows with inconsistent cost-overrun calculations:", (difference > 0.01).sum())

Rows with inconsistent cost-overrun calculations: 0


In [11]:
#Check project ID uniqueness

print("Total records:", len(df))
print("Unique project IDs:", df["Project_ID"].nunique())

Total records: 500
Unique project IDs: 500


In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

In [13]:
#Define the target variable

target = "Cost_Overrun_Cr"

In [14]:
#Select input variables available for the initial model

features = [
"Original_Cost_Cr",
"Sector",
"State",
"Implementing_Agency"
]

In [15]:
#Create model input and target

X = df[features].copy()
y = df[target].copy()

In [19]:
#Define numerical and categorical variables

numeric_features = ["Original_Cost_Cr"]

categorical_features = [
"Sector",
"State",
"Implementing_Agency"
]

In [16]:
#Prepare numerical variables

numeric_transformer = Pipeline(steps=[
("imputer", SimpleImputer(strategy="median"))
])

In [17]:
#Convert categories into numerical indicators

categorical_transformer = Pipeline(steps=[
("imputer", SimpleImputer(strategy="most_frequent")),
("onehot", OneHotEncoder(handle_unknown="ignore"))
])

In [20]:
#Combine the preprocessing steps

preprocessor = ColumnTransformer(transformers=[
("num", numeric_transformer, numeric_features),
("cat", categorical_transformer, categorical_features)
])

In [21]:
#Split the dataset

X_train, X_test, y_train, y_test = train_test_split(
X,
y,
test_size=0.20,
random_state=42
)

print("Data preparation completed successfully!")
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print("Input variables:", features)
print("Target variable:", target)

Data preparation completed successfully!
Training records: 400
Testing records: 100
Input variables: ['Original_Cost_Cr', 'Sector', 'State', 'Implementing_Agency']
Target variable: Cost_Overrun_Cr


In [22]:
#Build the Linear Regression model

model = Pipeline(steps=[
("preprocessor", preprocessor),
("regressor", LinearRegression())
])

In [23]:
#Train the model

model.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['Original_Cost_Cr']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['Sector', 'State',
                                                   'Implementing_Agency'])])),
                ('regressor', LinearRegression())])

In [24]:
#Predict cost overruns

y_pred = model.predict(X_test)

In [25]:
#Evaluate the model

print("LINEAR REGRESSION RESULTS")
print("-------------------------")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f} Cr")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f} Cr")
print(f"R-squared: {r2_score(y_test, y_pred):.3f}")

LINEAR REGRESSION RESULTS
-------------------------
MAE: 176.02 Cr
RMSE: 304.65 Cr
R-squared: 0.163


In [26]:
#Build a baseline model for comparison

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("\nBASELINE RESULTS")
print("-------------------------")
print(f"MAE: {mean_absolute_error(y_test, baseline_pred):.2f} Cr")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, baseline_pred)):.2f} Cr")
print(f"R-squared: {r2_score(y_test, baseline_pred):.3f}")


BASELINE RESULTS
-------------------------
MAE: 195.70 Cr
RMSE: 333.02 Cr
R-squared: -0.000


In [27]:
# Generate predictions for all projects
scored_df = df.copy()

# Predict cost overrun using the trained model
scored_df["Predicted_Cost_Overrun_Cr"] = model.predict(
    df[features]
)

# Calculate the difference between actual and predicted values
scored_df["Prediction_Error_Cr"] = (
    scored_df["Cost_Overrun_Cr"]
    - scored_df["Predicted_Cost_Overrun_Cr"]
)

# Save the results for Power BI
scored_df.to_csv(
    "Construction_Cost_Overrun_Predictions.csv",
    index=False
)

# Display a sample
display(
    scored_df[
        [
            "Project_ID",
            "Project_Name",
            "Cost_Overrun_Cr",
            "Predicted_Cost_Overrun_Cr",
            "Prediction_Error_Cr"
        ]
    ].head(10)
)

print("Prediction file created successfully!")

,Project_ID,Project_Name,Cost_Overrun_Cr,Predicted_Cost_Overrun_Cr,Prediction_Error_Cr
0,PRJ0001,Mega Infrastructure Development Phase 1,166.47,169.186132,-2.716132
1,PRJ0002,Mega Infrastructure Development Phase 2,0.00,221.877779,-221.877779
2,PRJ0003,Mega Infrastructure Development Phase 3,251.76,257.353343,-5.593343
3,PRJ0004,Mega Infrastructure Development Phase 4,0.00,423.650947,-423.650947
4,PRJ0005,Mega Infrastructure Development Phase 5,1024.40,177.853357,846.546643
5,PRJ0006,Mega Infrastructure Development Phase 6,557.44,274.835680,282.604320
6,PRJ0007,Mega Infrastructure Development Phase 7,139.61,159.018360,-19.408360
7,PRJ0008,Mega Infrastructure Development Phase 8,0.00,207.890969,-207.890969
8,PRJ0009,Mega Infrastructure Development Phase 9,550.27,228.311186,321.958814
9,PRJ0010,Mega Infrastructure Development Phase 10,373.80,190.130479,183.669521


Prediction file created successfully!


In [28]:
from google.colab import files

files.download("Construction_Cost_Overrun_Predictions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>